<h1 align="center">Analise</h1>

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline

In [ ]:
def resolve_support_images() -> Path:
    """Find support_images next to cwd or one level up (e.g. running from notebook/)."""
    cwd = Path.cwd().resolve()
    for base in (cwd, cwd.parent):
        candidate = base / "support_images"
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find support_images/. Run the notebook from the project root or from notebook/."
    )

In [ ]:
SUPPORT_IMAGES = resolve_support_images()
print(SUPPORT_IMAGES)

In [ ]:
def find_csv_files(root: Path) -> list[Path]:
    return sorted(root.rglob("*.csv"))

In [ ]:
csv_paths = find_csv_files(SUPPORT_IMAGES)
print(f"Found {len(csv_paths)} CSV file(s)")
for p in csv_paths:
    print(" ", p.relative_to(SUPPORT_IMAGES))

In [ ]:
def load_csv(path: Path, **read_csv_kwargs) -> pd.DataFrame:
    return pd.read_csv(path, **read_csv_kwargs)

In [ ]:
def load_all_csv(root: Path, **read_csv_kwargs) -> dict[str, pd.DataFrame]:
    out: dict[str, pd.DataFrame] = {}
    for path in find_csv_files(root):
        key = str(path.relative_to(root))
        out[key] = load_csv(path, **read_csv_kwargs)
    return out

In [ ]:
tables = load_all_csv(SUPPORT_IMAGES)
list(tables.keys())

In [ ]:
FIGSIZE = (10, 4)
EPOCH_COL = "epoch"
# Typical names from this project's logs: dice_test / loss_test (or dice_train / loss_train)
DICE_COL = "dice_test"
LOSS_COL = "loss_test"
DICE_COL_TRAIN = "dice_train"
LOSS_COL_TRAIN = "loss_train"
AGGREGATE_BY_EPOCH = True  # if True, mean Dice/Loss per epoch when multiple rows share an epoch


def plot_epoch_metrics(rel_path: str, df: pd.DataFrame) -> None:
    cols_needed = [EPOCH_COL, DICE_COL, LOSS_COL, DICE_COL_TRAIN, LOSS_COL_TRAIN]
    missing = [c for c in cols_needed if c not in df.columns]
    if missing:
        print(f"{rel_path}: missing columns {missing}. Available: {list(df.columns)}\n")
        return

    work = df[cols_needed].copy()
    for c in (DICE_COL, LOSS_COL, DICE_COL_TRAIN, LOSS_COL_TRAIN):
        work[c] = pd.to_numeric(work[c], errors="coerce")

    if AGGREGATE_BY_EPOCH:
        plot_df = work.groupby(EPOCH_COL, as_index=False)[[DICE_COL, LOSS_COL, DICE_COL_TRAIN, LOSS_COL_TRAIN]].mean().sort_values(EPOCH_COL)
    else:
        plot_df = work.sort_values(EPOCH_COL)

    epochs = plot_df[EPOCH_COL]

    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.plot(epochs, plot_df[DICE_COL], marker="o", markersize=3, linewidth=1, label="teste")
    ax.plot(epochs, plot_df[DICE_COL_TRAIN], marker="o", markersize=3, linewidth=1, label="treino")
    ax.set_title(f"{rel_path} — Dice vs epoch")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(DICE_COL)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.plot(epochs, plot_df[LOSS_COL], marker="o", markersize=3, linewidth=1, color="C1", label="teste")
    ax.plot(epochs, plot_df[LOSS_COL_TRAIN], marker="o", markersize=3, linewidth=1, label="treino")
    ax.set_title(f"{rel_path} — Loss vs epoch")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(LOSS_COL)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


if not tables:
    print("No CSV files under support_images yet. Add some, then re-run the discovery cells.")
else:
    for rel_path, df in tables.items():
        print(f"\n--- {rel_path} ---")
        display(df.head())
        plot_epoch_metrics(rel_path, df)
